In [1]:
import os

In [2]:
%pwd

'd:\\End_to_End_ML_project_for_Ad_Click_Through_Rate_Prediction\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'd:\\End_to_End_ML_project_for_Ad_Click_Through_Rate_Prediction'

In [5]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class ModelTrainingConfig:
    root_dir: Path
    train_file_path: Path
    validation_file_path: Path
    model_file_path: Path
    metrics_file_path: Path
    model_params: dict
    target_column: str

In [6]:
from src.ad_ctr_prediction.constants import *
from src.ad_ctr_prediction.utils.common import read_yaml, create_directories

In [7]:
class ConfigurationManager:
    def __init__(self, config_filepath = CONFIG_FILE_PATH, params_filepath = PARAMS_FILE_PATH, schema_filepath = SCHEMA_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    def get_model_training_config(self) -> ModelTrainingConfig:
        config = self.config.model_training

        create_directories([config.root_dir])

        model_training_config = ModelTrainingConfig(
            root_dir=config.root_dir,
            train_file_path=config.train_file_path,
            validation_file_path=config.validation_file_path,
            model_file_path=config.model_file_path,
            metrics_file_path=config.metrics_file_path,
            model_params=dict(self.params.model_params),
            target_column=config.target_column
        )

        return model_training_config    

In [8]:
import pandas as pd
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, log_loss, precision_recall_curve, auc
from src.ad_ctr_prediction import logger
from src.ad_ctr_prediction.utils.common import save_bin, save_json

In [11]:
from pathlib import Path

class ModelTraining:
    def __init__(self, config: ModelTrainingConfig):
        self.config = config

    def load_split_data(self, file_path: str) -> pd.DataFrame:
        if not os.path.exists(file_path):
            raise FileNotFoundError(f"Split data file not found: {file_path}")

        logger.info(f"Loading split data from: {file_path}")
        return pd.read_csv(file_path)

    def train_model(self, train_df: pd.DataFrame, val_df: pd.DataFrame) -> XGBClassifier:
        target_col = self.config.target_column
        if target_col not in train_df.columns or target_col not in val_df.columns:
            raise KeyError(f"Target column '{target_col}' not found in train/validation data")

        X_train = train_df.drop(columns=[target_col])
        y_train = train_df[target_col]
        X_val = val_df.drop(columns=[target_col])
        y_val = val_df[target_col]

        logger.info("Initializing XGBoost classifier")
        model = XGBClassifier(**self.config.model_params)

        logger.info("Starting XGBoost training with early stopping")
        model.fit(
            X_train,
            y_train,
            eval_set=[(X_val, y_val)],
            verbose=False
        )

        logger.info("Model training completed")
        return model

    def evaluate_model(self, model: XGBClassifier, df: pd.DataFrame) -> dict:
        target_col = self.config.target_column
        X = df.drop(columns=[target_col])
        y = df[target_col]

        y_pred = model.predict(X)
        y_proba = model.predict_proba(X)[:, 1] if hasattr(model, "predict_proba") else None
        
        precision, recall, _ = precision_recall_curve(y, y_proba) if y_proba is not None else (None, None, None)
        pr_auc = auc(recall, precision) if precision is not None and recall is not None else None

        metrics = {
            "accuracy": accuracy_score(y, y_pred),
            "precision": precision_score(y, y_pred, zero_division=0),
            "recall": recall_score(y, y_pred, zero_division=0),
            "f1_score": f1_score(y, y_pred, zero_division=0),
            "roc_auc_score": roc_auc_score(y, y_proba) if y_proba is not None else None,
            "log_loss": log_loss(y, y_proba) if y_proba is not None else None,
            "pr_auc": pr_auc
        }

        return {k: float(v) for k, v in metrics.items()}

    def save_model(self, model: XGBClassifier) -> None:
        model_path = Path(self.config.model_file_path)
        model_path.parent.mkdir(parents=True, exist_ok=True)
        save_bin(data=model, path=model_path)
        logger.info(f"Saved trained model at: {model_path}")

    def save_metrics(self, metrics: dict) -> None:
        metrics_path = Path(self.config.metrics_file_path)
        metrics_path.parent.mkdir(parents=True, exist_ok=True)
        save_json(path=metrics_path, data=metrics)
        logger.info(f"Saved training metrics at: {metrics_path}")

    def initiate_model_training(self) -> bool:
        try:
            train_df = self.load_split_data(self.config.train_file_path)
            val_df = self.load_split_data(self.config.validation_file_path)
            # test_df = self.load_split_data(self.config.test_file_path)

            model = self.train_model(train_df, val_df)
            self.save_model(model)

            logger.info("Evaluating model on validation data")
            validation_metrics = self.evaluate_model(model, val_df)
            # test_metrics = self.evaluate_model(model, test_df)

            metrics = {
                "validation": validation_metrics,
                # "test": test_metrics,
                "best_iteration": int(model.get_booster().best_iteration) if hasattr(model, "get_booster") and model.get_booster().best_iteration != -1 else None
            }

            self.save_metrics(metrics)
            logger.info("Model training stage completed successfully")
            return True
        except Exception as e:
            logger.error(f"Error during model training: {str(e)}")
            raise e


In [12]:
try:
    config = ConfigurationManager()
    model_trainer_config = config.get_model_training_config()
    model_trainer_config = ModelTraining(config=model_trainer_config)
    model_trainer_config.initiate_model_training()
except Exception as e:
    raise e

[2026-06-20 20:04:20,306: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-06-20 20:04:20,314: INFO: common: yaml file: params.yaml loaded successfully]
[2026-06-20 20:04:20,370: INFO: common: yaml file: config\schema.yaml loaded successfully]
[2026-06-20 20:04:20,380: INFO: common: created directory at: artifacts]
[2026-06-20 20:04:20,386: INFO: common: created directory at: artifacts/model_training]
[2026-06-20 20:04:20,393: INFO: 2887539844: Loading split data from: artifacts/data_transformation/split_data/train.csv]
[2026-06-20 20:04:22,515: INFO: 2887539844: Loading split data from: artifacts/data_transformation/split_data/validation.csv]
[2026-06-20 20:04:22,999: INFO: 2887539844: Initializing XGBoost classifier]
[2026-06-20 20:04:22,999: INFO: 2887539844: Starting XGBoost training with early stopping]
[2026-06-20 20:04:38,680: INFO: 2887539844: Model training completed]
[2026-06-20 20:04:38,744: INFO: common: binary file saved at: artifacts\model_training\x